In [1]:
import pandas as pd
from pathlib import Path
from pptoolbox.interpolate import get_interpolated_data

In [ ]:
raw_folder_path = Path("../../data/raw")

datasets = {}

for folder in raw_folder_path.iterdir():
    if folder.name != "may2026pull":
        continue
    print(folder.name)
    
    # extract info_*.csv and spectra_*.csv files
    info_file = list(folder.glob("info_*.csv"))
    spectra_file = list(folder.glob("spectra_*.csv"))
    if info_file and spectra_file:
        info_df = pd.read_csv(info_file[0])
        spectra_df = pd.read_csv(spectra_file[0])
        datasets[folder.name] = {"info": info_df, "spectra": spectra_df}

may2026pull


In [3]:
for dataset in datasets:
    label = datasets[dataset]["info"].pivot(index=['lot_id'], columns='property_name', values='property_value').reset_index()
    datasets[dataset]["label"] = label
    display(label.head())

property_name,lot_id,Any_Adulteration,Brix,BrownSug_Adulteration,Farm,Grade,Preservative,Viscosity,pH
0,116565,NaN,67.2,PureKD,PWK,NS,Chalk,3,6.6
1,116566,NaN,69.2,Adulterated,PWK,A,Sulfite,3.8,5.4
2,116567,NaN,67.2,Adulterated,PWK,NS,Chalk,3,6.6
3,116568,NaN,67.2,Adulterated,PWK,NS,Chalk,3,6.6
4,116569,NaN,66.2,Adulterated,PWK,NS,Chalk,2.8,7


In [4]:
for dataset in datasets:
    print (f"Dataset: {dataset}")
    print (datasets[dataset]["label"].isna().sum())

Dataset: may2026pull
property_name
lot_id                     0
Any_Adulteration         914
Brix                     335
BrownSug_Adulteration      0
Farm                       0
Grade                      0
Preservative               0
Viscosity                335
pH                       335
dtype: int64


In [5]:
columns_to_keep = ['lot_id', 'BrownSug_Adulteration', 'Farm', 'Grade', 'Preservative', 'pH', 'Brix', 'Viscosity']
for dataset in datasets:
    datasets[dataset]["label"] = datasets[dataset]["label"][columns_to_keep]
    print (f"Dataset: {dataset}")
    print (datasets[dataset]["label"].isna().sum())

Dataset: may2026pull
property_name
lot_id                     0
BrownSug_Adulteration      0
Farm                       0
Grade                      0
Preservative               0
pH                       335
Brix                     335
Viscosity                335
dtype: int64


In [6]:
for dataset in datasets:
    label = datasets[dataset]["label"]
    spectra = datasets[dataset]["spectra"]

    label_lot_id = set(label['lot_id'])
    spectra_lot_id = set(spectra['lot_id'])

    missing_label_lot_ids = spectra_lot_id - label_lot_id
    print(f"Missing lot IDs in label for {dataset}: {missing_label_lot_ids}")

    missing_spectra_lot_ids = label_lot_id - spectra_lot_id
    print(f"Missing lot IDs in spectra for {dataset}: {missing_spectra_lot_ids}")

Missing lot IDs in label for may2026pull: set()
Missing lot IDs in spectra for may2026pull: set()


In [7]:
# concat all datasets together
all_labels = pd.concat([datasets[dataset]["label"] for dataset in datasets], ignore_index=True)
all_spectra = pd.concat([datasets[dataset]["spectra"] for dataset in datasets], ignore_index=True)

# Preprocessing

In [8]:
X, y, add = get_interpolated_data(all_spectra, all_labels, export_meta=True)

meta = add['metadata']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Metadata shape: {meta.shape}")

X shape: (5950, 191)
y shape: (5950, 7)
Metadata shape: (5950, 4)


In [9]:
y.head()

,BrownSug_Adulteration,Farm,Grade,Preservative,pH,Brix,Viscosity
lot_id,,,,,,,
116565,PureKD,PWK,NS,Chalk,6.6,67.2,3
116565,PureKD,PWK,NS,Chalk,6.6,67.2,3
116565,PureKD,PWK,NS,Chalk,6.6,67.2,3
116565,PureKD,PWK,NS,Chalk,6.6,67.2,3
116565,PureKD,PWK,NS,Chalk,6.6,67.2,3


In [10]:
meta[['Farm','Preservative']] = y[['Farm','Preservative']]
y = y.drop(columns=['Farm','Preservative'])
display(meta.head())
display(y.head())

,analyzer_id,specimen_id,date_scanned,lot_name,Farm,Preservative
lot_id,,,,,,
116565,82,700881,1768458344,15012026-B9833BIS,PWK,Chalk
116565,82,700882,1768458366,15012026-B9833BIS,PWK,Chalk
116565,82,700883,1768458387,15012026-B9833BIS,PWK,Chalk
116565,82,700884,1768458408,15012026-B9833BIS,PWK,Chalk
116565,82,700885,1768458430,15012026-B9833BIS,PWK,Chalk


,BrownSug_Adulteration,Grade,pH,Brix,Viscosity
lot_id,,,,,
116565,PureKD,NS,6.6,67.2,3
116565,PureKD,NS,6.6,67.2,3
116565,PureKD,NS,6.6,67.2,3
116565,PureKD,NS,6.6,67.2,3
116565,PureKD,NS,6.6,67.2,3


In [11]:
meta['date_scanned_reformatted'] = pd.to_datetime(meta['date_scanned'], unit='s').dt.strftime('%Y-%m-%d')

## filter off bad data from 11 apr to 12 may

In [12]:
def filter_off_bad_data(X, y, meta):
    # Convert to datetime if it's a string
    dates = pd.to_datetime(meta['date_scanned_reformatted'])
    mask = (dates < '2026-04-11') | (dates > '2026-05-12')
    return X[mask], y[mask], meta[mask]

In [13]:
X_masked, y_masked, meta_masked = filter_off_bad_data(X, y, meta)
print(f"X_masked shape: {X_masked.shape}")
print(f"y_masked shape: {y_masked.shape}")
print(f"meta_masked shape: {meta_masked.shape}")

X_masked shape: (4326, 191)
y_masked shape: (4326, 5)
meta_masked shape: (4326, 7)


In [14]:
meta_masked['date_scanned_reformatted'].unique()

array(['2026-01-15', '2026-01-19', '2026-01-20', '2026-01-21',
       '2026-01-22', '2026-01-23', '2026-01-24', '2026-01-26',
       '2026-01-27', '2026-01-28', '2026-01-29', '2026-01-30',
       '2026-01-31', '2026-02-02', '2026-02-03', '2026-02-04',
       '2026-02-05', '2026-02-06', '2026-02-07', '2026-02-09',
       '2026-02-10', '2026-02-11', '2026-02-12', '2026-02-13',
       '2026-02-14', '2026-02-16', '2026-02-18', '2026-02-19',
       '2026-02-20', '2026-02-21', '2026-02-23', '2026-02-24',
       '2026-02-25', '2026-02-26', '2026-02-27', '2026-02-28',
       '2026-03-02', '2026-03-03', '2026-03-04', '2026-03-05',
       '2026-03-06', '2026-03-07', '2026-03-09', '2026-03-10',
       '2026-03-11', '2026-03-12', '2026-03-13', '2026-03-14',
       '2026-03-16', '2026-03-17', '2026-03-25', '2026-03-26',
       '2026-03-27', '2026-03-28', '2026-03-30', '2026-03-31',
       '2026-04-01', '2026-04-02', '2026-04-04', '2026-04-06',
       '2026-04-07', '2026-04-08', '2026-04-09', '2026-

In [16]:
X, y, meta = X_masked, y_masked, meta_masked

In [17]:
# show meta where y isna
meta[y['BrownSug_Adulteration'].isna()].drop_duplicates(subset=['lot_name'])

,analyzer_id,specimen_id,date_scanned,lot_name,Farm,Preservative,date_scanned_reformatted
lot_id,,,,,,,


In [18]:
meta[y['Grade'].isna()].drop_duplicates(subset=['lot_name'])

,analyzer_id,specimen_id,date_scanned,lot_name,Farm,Preservative,date_scanned_reformatted
lot_id,,,,,,,


In [19]:
meta.isna().sum()

analyzer_id                 0
specimen_id                 0
date_scanned                0
lot_name                    0
Farm                        0
Preservative                0
date_scanned_reformatted    0
dtype: int64

In [20]:
mask = y['BrownSug_Adulteration'] != 'Adulterated'

X = X[mask]
y = y[mask]
meta = meta[mask]

In [21]:
assert X.index.equals(y.index), "Indices of X and y do not match!"
assert X.index.equals(meta.index), "Indices of X and meta do not match!"

In [22]:
y[~y.index.duplicated(keep='first')].value_counts('Grade')

Grade
NS    321
C      70
B      68
A      54
A3      1
Name: count, dtype: int64

In [23]:
y[~y.index.duplicated(keep='first')].value_counts('BrownSug_Adulteration')

BrownSug_Adulteration
PureKD    514
Name: count, dtype: int64

In [35]:
meta.drop_duplicates(subset=['lot_name'])

,analyzer_id,specimen_id,date_scanned,lot_name,Farm,Preservative,date_scanned_reformatted
lot_id,,,,,,,
116565,82,700881,1768458344,15012026-B9833BIS,PWK,Chalk,2026-01-15
117235,82,704242,1768921685,20012026-B9234BIS,PWK,Chalk,2026-01-20
117236,82,704237,1768921540,20012026-R8336OB,PWK,Chalk,2026-01-20
117237,82,704232,1768921377,20012026-B9833BIS,PWK,Chalk,2026-01-20
117238,82,704227,1768921115,20012026-B9236BIS,PWK,Chalk,2026-01-20
...,...,...,...,...,...,...,...
124948,82,764216,1775804097,100426_B9236BIS,PWK,Chalk,2026-04-10
124949,82,764221,1775804515,100426_B9238BIS,PWK,Chalk,2026-04-10
124950,82,764226,1775804741,100426_R9718MK,PWK,Sulfite,2026-04-10


In [36]:
y[['pH','Brix','Viscosity']].isna().sum()

pH           1250
Brix         1250
Viscosity    1250
dtype: int64

In [34]:
has_na = y[['pH','Brix','Viscosity']].isna().any(axis=1)
meta[has_na].drop_duplicates(subset=['lot_name'])

,analyzer_id,specimen_id,date_scanned,lot_name,Farm,Preservative,date_scanned_reformatted
lot_id,,,,,,,
117235,82,704242,1768921685,20012026-B9234BIS,PWK,Chalk,2026-01-20
117236,82,704237,1768921540,20012026-R8336OB,PWK,Chalk,2026-01-20
117237,82,704232,1768921377,20012026-B9833BIS,PWK,Chalk,2026-01-20
117238,82,704227,1768921115,20012026-B9236BIS,PWK,Chalk,2026-01-20
117239,82,704222,1768920733,20012026-B9263BIS,PWK,Chalk,2026-01-20
...,...,...,...,...,...,...,...
119753,82,723211,1771399674,180226_A8361ZF,PWK,Chalk,2026-02-18
119896,82,723963,1771475629,190226_R8042IJ,PWK,Sulfite,2026-02-19
119897,82,723968,1771475866,190226_R8669JJ,PWK,Sulfite,2026-02-19


In [37]:
X = X[~has_na]
y = y[~has_na]
meta = meta[~has_na]

print (f"Final X shape: {X.shape}")
print (f"Final y shape: {y.shape}")
print (f"Final meta shape: {meta.shape}")

Final X shape: (1320, 191)
Final y shape: (1320, 5)
Final meta shape: (1320, 7)


In [ ]:
output_path = Path ("../../data/processed") / 'v4' / "2026_jan_apr_b4fail"
output_path.mkdir(parents=True, exist_ok=True)
X.to_csv(output_path / f"input.csv")
y.to_csv(output_path / f"label.csv")
meta.to_csv(output_path / f"meta.csv")

# combine with previous set of data and helena scans in jan 2026

In [ ]:
version = "v2"

full_folder = Path("../data/processed") / version / "full"

X_full = pd.read_csv(full_folder / "input.csv", index_col=0)
y_full = pd.read_csv(full_folder / "label.csv", index_col=0)
meta_full = pd.read_csv(full_folder / "meta.csv", index_col=0)
meta_full = meta_full.rename(columns={'date_scanned':'date_scanned_reformatted'})

print(f"Shapes of X, y, meta: {X_full.shape}, {y_full.shape}, {meta_full.shape}")

In [ ]:
meta_full.tail()

In [ ]:
helena_folder = Path("../data/processed") / version / "eval_helena" /"full"

X_helena = pd.read_csv(helena_folder / "input.csv", index_col=0)
y_helena = pd.read_csv(helena_folder / "label.csv", index_col=0)
meta_helena = pd.read_csv(helena_folder / "meta.csv", index_col=0)
meta_helena['set'] = 'helena'
meta_helena = meta_helena.rename(columns={'date_scanned':'date_scanned_reformatted'})

print(f"Shapes of X, y, meta: {X_helena.shape}, {y_helena.shape}, {meta_helena.shape}")

In [ ]:
mask_helena = meta_helena['lot_name'].str.split(' ').str[0] != 'Adulteration'
print(mask_helena.sum())

X_helena = X_helena[mask_helena]
y_helena = y_helena[mask_helena]
meta_helena = meta_helena[mask_helena]
meta_helena.tail(5)

In [ ]:
print(f'Shapes of X, y, meta: {X.shape}, {y.shape}, {meta.shape}')
meta = meta.copy()
meta['set'] = 'wh2026'
meta['date_scanned'] = meta['date_scanned'].astype(str)

In [ ]:
X

In [ ]:
X.columns = X.columns.astype(str)

In [ ]:
X_combined = pd.concat([X_full, X_helena, X])
y_combined = pd.concat([y_full, y_helena, y.loc[:,y_full.columns]])
meta_combined = pd.concat([meta_full, meta_helena, meta])

print(f'Shapes of combined X, y, meta: {X_combined.shape}, {y_combined.shape}, {meta_combined.shape}')

In [ ]:
meta_combined

In [ ]:
meta_combined.isna().sum()

In [ ]:
y_combined[~y_combined.index.duplicated(keep='first')].value_counts()

In [ ]:
assert X_combined.index.equals(y_combined.index), "Indices of combined X and y do not match!"
assert X_combined.index.equals(meta_combined.index), "Indices of combined X and meta do not match!"
assert X_combined.index.is_monotonic_increasing

In [ ]:
v3_output_path = Path ("../data/processed") / 'v3'
v3_output_path.mkdir(parents=True, exist_ok=True)

full_folder = v3_output_path / "full"
full_folder.mkdir(parents=True, exist_ok=True)

X_combined.to_csv(full_folder / f"input.csv")
y_combined.to_csv(full_folder / f"label.csv")
meta_combined.to_csv(full_folder / f"meta.csv")